In [44]:
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd
import numpy as np
import os
import signal
import sys

In [174]:
def getStatsBowling(keyword,year):
    r=requests.get(f'https://www.iplt20.com/stats/{year}/{keyword}?stats_type=bowling')
    #r=requests.get("https://www.iplt20.com/stats/2021/mostRuns?stats_type=batting")
    soup=BeautifulSoup(r.text,'html.parser')
    table=soup.find('div',{"id":"bowlingTAB"})
    headers=[]
    information=[]
    flag=True
    for elem in table.find_all('tr'):
        player_info=[]
        for row in elem.find_all('th'):
            headers.append(row.text)
        if flag==True:
            headers.append("Year")
            flag=False
        for row in elem.find_all('td'):
            player_info.append(row.text)
        player_info.append(year)
        information.append(player_info)
    df = pd.DataFrame(information[1:], columns = headers)
    df['Player'] = df['Player'].apply(lambda x: x.strip())
    return df

In [48]:
#getStats("mostWkts","2021")
dfs=[]
bowlingKeywords=["mostWkts","bestBowlEco","mostDotBalls","bestBowlAvg"]
years=["2021","2020","2019","2018"]
for year in years :
    dfs.append(getStatsBowling("mostWkts",year))
finaldf=pd.concat(dfs)
finaldf.to_csv("bowlingstats.csv",index=False)

In [36]:
def getStats(keyword,year,stats_type):
    r=requests.get(f'https://www.iplt20.com/stats/{year}/{keyword}?stats_type={stats_type}')
    #r=requests.get("https://www.iplt20.com/stats/2021/mostRuns?stats_type=batting")
    soup=BeautifulSoup(r.text,'html.parser')
    table=soup.find('table',class_='np-mostruns_table')
    headers=[]
    information=[]
    flag=True
    for elem in table.find_all('tr'):
        player_info=[]
        for row in elem.find_all('th'):
            headers.append(row.text)
        if flag==True:
            headers.append("Year")
            flag=False
        for row in elem.find_all('td'):
            player_info.append(row.text)
        player_info.append(year)
        information.append(player_info)
    df = pd.DataFrame(information[1:], columns = headers)
    df['Player'] = df['Player'].apply(lambda x: x.strip())
    return df

In [58]:
battingKeywords=["mostRuns","most4s","most6s","most50s","bestStrikeRate","bestbattingaverage","highestScores"]

years=["2021","2020","2019","2018"]
dfs=[]
for year in years:
    df=getStats("mostRuns",year,"batting")
    dfs.append(df)
finaldf=pd.concat(dfs)
finaldf.to_csv("BattingStats.csv",index=False)
    

In [109]:
cricmetric=pd.read_csv("battingcricmetric.csv")
cricmetric = cricmetric.rename(columns={'Player': 'cricmetricName'})
cricmetric = cricmetric.astype("string")
cricmetric.dtypes
cricmetric


,cricmetricName,I,R,B,Outs,Avg,SR,HS,4s,6s,50,100
0,AB de Villiers,14,313,211,10,31.3,148.34,76,23,16,2,0
1,Abdul Samad,10,111,87,9,12.33,127.59,28,4,8,0,0
2,Abhishek Sharma,7,98,75,6,16.33,130.67,33,7,4,0,0
3,AD Russell,8,183,120,7,26.14,152.5,54,14,14,1,0
4,AF Milne,3,16,17,2,8.0,94.12,15,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
144,VR Iyer,10,370,288,9,41.11,128.47,67,37,14,4,0
145,Washington Sundar,4,31,47,4,7.75,65.96,10,2,0,0,0
146,WP Saha,9,131,140,9,14.56,93.57,44,7,5,0,0
147,YBK Jaiswal,10,249,168,10,24.9,148.21,50,32,10,1,0


In [137]:
salaries=pd.read_csv("iplSalary2021.csv")

In [148]:
result=pd.merge(cricmetric,salaries,left_on=["cricmetricName"],right_on=["Player"],how="left").dropna()
result.to_csv("salariesFinalBatting.csv")

In [177]:
def createSalariesFile(metricfile,salariesfile,year):
    bowlingCricMetric=pd.read_csv(metricfile)
    salaries=pd.read_csv(salariesfile)
    result=pd.merge(bowlingCricMetric,salaries,on="Player",how="left").dropna()
    reqColumns=list(bowlingCricMetric.columns)
    reqColumns.append("Color")
    result=result[reqColumns]
    result=result.rename(columns={"Color":"Salary"})
    result['Year']=year
    return result

dfs=[]
years=["2021","2020","2019","2018"]
typeofStat=['batting','bowling']
for stat in typeofStat:
    for year in years:
        metricfile=f"./pricePredictionDatasets/{stat}cricmetric{year}.csv"
        salariesfile=f"./pricePredictionDatasets/iplSalary{year}.csv"
        dfs.append(createSalariesFile(metricfile,salariesfile,year))
    salariesdf=pd.concat(dfs)
    salariesdf.to_csv(f"{stat}Salaries.csv",index=False)
    dfs=[]




,Player,I,O,R,W,Econ,Avg,SR,4W,5W,BBI,4s,6s,Dots,Salary,year
0,A Mishra,4,14.0,109,6,7.79,18.17,14.00,1,0,4/24,6,4,25,"625,000",2020
1,A Nortje,8,30.2,187,12,6.16,15.58,15.17,0,0,2/12,17,2,84,"28,600",2020
2,Abdul Samad,1,1.0,9,1,9.00,9.00,6.00,0,0,1/9,1,0,1,"28,600",2020
3,Abhishek Sharma,6,10.0,64,4,6.40,16.00,15.00,0,0,2/4,5,1,25,"85,940",2020
4,AD Russell,8,19.0,188,11,9.89,17.09,10.36,0,1,5/15,18,8,39,"1,328,130",2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,TK Curran,3,11.2,104,4,9.18,26.00,17.00,0,0,3/29,9,4,19,"750,000",2020
106,V Shankar,5,11.0,100,3,9.09,33.33,22.00,0,0,2/19,6,5,22,"500,000",2020
107,VR Iyer,4,8.3,69,3,8.12,23.00,17.00,0,0,2/29,3,3,15,"28,600",2020
108,Washington Sundar,6,16.0,118,3,7.38,39.33,32.00,0,0,1/7,9,4,32,"500,000",2020
